# 04d. 증강 로지스틱 회귀 - 변이 랜덤 삭제

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: test에서 변이가 덜 잡힌 환자에게 덜 깎이는지 확인 (오늘 계획 제출 순서 2번 "가설 확인용")
- 모델·피처: 로지스틱 회귀 E (C=0.1, class_weight=balanced, max_iter=3000), ② make_features_E와 같은 피처
- 증강: 학습 행마다 변이를 확률 p(0.5 / 0.7 / 0.9)로 남긴 사본 3개를 만들어 원본과 함께 학습. 검증 행은 그대로
- 누수 방지: 사본은 모델 학습(fit) 안에서만 만들어서, run_cv의 fold 학습 부분으로만 생성
- 기록 파일: `experiments/log_P_light.csv` (`1110_logreg_E_aug_gkf`)
- 예측 파일: `experiments/preds/P_light__logreg_E_aug_1110__*`
- ⚠️ Run All 하면 "5. 교차검증"이 다시 기록돼 기록표에 같은 줄이 또 생겨요.

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [10]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


In [11]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

OS: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18 | pandas 3.1.1 | scikit-learn 1.9.1


## 1. 도구·데이터 불러오기

In [12]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, vstack
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.features import (mutation_count, mutation_type_counts,
                          amino_acid_pair_counts, gene_truncation_flags)
from src.validation import get_folds, run_cv
from src.evaluation import log_result, save_predictions

train, test = load_data(DATA_DIR)   # test는 마지막 예측에만 사용
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                # 팀 공통 그룹 K-fold
print("train", train.shape, "| 암종 수", len(le.classes_), "| fold 수", len(folds))


train (6201, 4386) | 암종 수 26 | fold 수 5


## 2. E 피처 — ② make_features_E와 같은 계산

In [13]:
def make_features_E(X):
    """② E 조합 피처. 한 환자 안에서만 계산."""
    return pd.concat([binarize_mutations(X),
                      np.log1p(mutation_count(X)),
                      np.log1p(mutation_type_counts(X)),
                      np.log1p(amino_acid_pair_counts(X)),
                      gene_truncation_flags(X)], axis=1)

def make_features_E_sparse(X):
    return csr_matrix(make_features_E(X).to_numpy(dtype=np.float32))

## 3. 증강 함수 — 변이를 확률 p로 남기고 나머지 삭제
한 칸에 변이가 여러 개면 각각 따로 판단하고, 모두 지워지면 WT. 결측 칸은 그대로. 원본은 바꾸지 않음.

In [14]:
def drop_mutations(X, keep_prob, seed):
    """변이 표기 하나하나를 keep_prob 확률로 남긴 사본. 다 지워진 칸은 WT."""
    rng = np.random.default_rng(seed)
    cells = X.stack().dropna()
    cells = cells[cells != "WT"]                                  # 변이가 있는 칸
    tokens = cells.astype(str).str.split().explode()              # 한 칸의 여러 변이를 하나씩
    kept = tokens[rng.random(len(tokens)) < keep_prob]
    kept = kept.groupby(level=[0, 1]).agg(" ".join)               # 남은 변이를 칸별로 다시 합침

    values = X.to_numpy(dtype=object).copy()
    rows = X.index.get_indexer(cells.index.get_level_values(0))
    cols = X.columns.get_indexer(cells.index.get_level_values(1))
    values[rows, cols] = "WT"                                     # 변이 칸을 일단 WT로
    rows = X.index.get_indexer(kept.index.get_level_values(0))
    cols = X.columns.get_indexer(kept.index.get_level_values(1))
    values[rows, cols] = kept.to_numpy()                          # 남은 변이만 되돌림
    return pd.DataFrame(values, index=X.index, columns=X.columns)

# 확인: 남은 변이 비율이 p 근처인지
sample = X.iloc[:300]
for p in (0.5, 0.7, 0.9):
    before = binarize_mutations(sample).to_numpy().sum()
    after = binarize_mutations(drop_mutations(sample, p, SEED)).to_numpy().sum()
    print(f"p={p}: 변이 칸 {before} → {after} ({after / before:.2f})")


p=0.5: 변이 칸 10789 → 5623 (0.52)
p=0.7: 변이 칸 10789 → 7781 (0.72)
p=0.9: 변이 칸 10789 → 9805 (0.91)


## 4. 증강 로지스틱 회귀 모델
학습(fit)할 때만 사본 3개를 만들어 원본과 함께 학습하고, 예측할 때는 입력을 그대로 사용.

In [15]:
class AugmentedLR(BaseEstimator, ClassifierMixin):
    """학습 행 + 변이 삭제 사본(keep_probs별 1개씩)으로 학습하는 로지스틱 회귀 E."""
    def __init__(self, keep_probs=(0.5, 0.7, 0.9), C=0.1, max_iter=3000, seed=SEED):
        self.keep_probs = keep_probs
        self.C = C
        self.max_iter = max_iter
        self.seed = seed

    def fit(self, X, y):
        y = np.asarray(y)
        parts = [make_features_E_sparse(X)]
        for i, p in enumerate(self.keep_probs):
            parts.append(make_features_E_sparse(drop_mutations(X, p, self.seed + i)))
        Z = vstack(parts).tocsr()
        y_all = np.tile(y, len(parts))                            # 사본의 라벨은 원본과 같음
        self.model_ = LogisticRegression(C=self.C, class_weight="balanced",
                                         max_iter=self.max_iter).fit(Z, y_all)
        self.classes_ = self.model_.classes_
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(make_features_E_sparse(X))

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

AUG_PARAMS = dict(keep_probs=(0.5, 0.7, 0.9), C=0.1, max_iter=3000, seed=SEED)
print("모델 준비 완료:", AUG_PARAMS)


모델 준비 완료: {'keep_probs': (0.5, 0.7, 0.9), 'C': 0.1, 'max_iter': 3000, 'seed': 42}


## 5. 교차검증 — 증강 LR (그룹 K-fold)
비교 기준: 증강 없는 LR E (② E_aa_pairs_truncation_group_lr, CV 0.4919). 0.01~0.02 하락은 예상 범위.

In [17]:
cv_aug = run_cv(AugmentedLR(**AUG_PARAMS), X, y, folds=folds)

log_result(owner="P_light", notebook="04d_augmented_lr.ipynb",
           experiment="1110_logreg_E_aug_gkf",
           features="E + 학습 행 변이 랜덤 삭제 사본 3개 (p=0.5/0.7/0.9)",
           model="LogisticRegression", params=AUG_PARAMS, cv=cv_aug,
           notes="증강 LR, 그룹 K-fold, 비교 기준 LR E 0.4919")
print("증강 없는 LR E 대비:", round(cv_aug["mean"] - 0.4919, 4))

fold 1: Macro F1 = 0.4791
fold 2: Macro F1 = 0.4818
fold 3: Macro F1 = 0.4686
fold 4: Macro F1 = 0.4941
fold 5: Macro F1 = 0.4965
CV Macro F1 = 0.4840 (std 0.0102)
기록: log_P_light.csv ← 1110_logreg_E_aug_gkf (CV 0.4840)
증강 없는 LR E 대비: -0.0079


## 6. 최종 학습 — train 전체 + 사본 3개, 예측 파일 저장

In [18]:
final_aug = AugmentedLR(**AUG_PARAMS).fit(X, y)
test_proba_aug = final_aug.predict_proba(test[X.columns])   # test는 예측에만 사용

save_predictions(owner="P_light", name="logreg_E_aug_1110",
                 oof_proba=cv_aug["oof_proba"], test_proba=test_proba_aug,
                 classes=le.classes_, cv=cv_aug,
                 notes="증강 LR (p=0.5/0.7/0.9), 그룹 K-fold")


예측 저장: P_light__logreg_E_aug_1110 (CV 0.4840) → experiments/preds/


## 7. 제출 파일 — 증강 XGBoost(③) + 증강 LR 50:50 (제출 2번)
> ⚠️ 다시 열었을 때: 1번 → 7번만 실행 (5·6번은 기록표·예측 파일이 중복 저장됨)

In [ ]:
import json
from src.config import EXPERIMENT_DIR
from src.prediction import make_submission

XGB_AUG_NAME = "A_model__xgb_E_depth3_invsqrt_mutdrop_050709_group_full_1110"   # ← ③ 증강 XGBoost

pred_dir = EXPERIMENT_DIR / "preds"
xgb_meta = json.load(open(pred_dir / f"{XGB_AUG_NAME}__meta.json"))
assert [str(c) for c in xgb_meta["classes"]] == [str(c) for c in le.classes_], "암종 순서가 달라요"
xgb_test = np.load(pred_dir / f"{XGB_AUG_NAME}__test.npy")
xgb_oof = np.load(pred_dir / f"{XGB_AUG_NAME}__oof.npy")

lr_oof = np.load(pred_dir / "P_light__logreg_E_aug_1110__oof.npy")    # 6번에서 저장한 증강 LR 예측
lr_test = np.load(pred_dir / "P_light__logreg_E_aug_1110__test.npy")

ens_oof = 0.5 * xgb_oof + 0.5 * lr_oof
ens_test = 0.5 * xgb_test + 0.5 * lr_test
from src.evaluation import macro_f1
print("증강 앙상블 CV(OOF 전체):", round(macro_f1(y, ens_oof.argmax(axis=1)), 4))

make_submission(test, le.inverse_transform(ens_test.argmax(axis=1)),
                "submission_aug_xgb_lr_1110.csv")

증강 앙상블 CV(OOF 전체): 0.5005
저장: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/submissions/submission_aug_xgb_lr_1110.csv (2546행)


,ID,SUBCLASS
0,TEST_0000,STES
1,TEST_0001,STES
2,TEST_0002,THCA
3,TEST_0003,LGG
4,TEST_0004,LUSC
...,...,...
2541,TEST_2541,STES
2542,TEST_2542,PAAD
2543,TEST_2543,DLBC
2544,TEST_2544,LAML
